In [ ]:
import torch
from torch.utils.data import DataLoader
import lightning as L
from lightning.fabric import seed_everything
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append('..')
from model import TransformerHIM
from dataset import SeqTeleopDataset

%load_ext autoreload
%autoreload 2

In [ ]:
seed_everything(42)
M = 2
D = 10
H = 128
O = 4

batch_size = 1 # Force batch size to 1 for now
max_epochs = 50

In [ ]:
setting = 'task_[0.001 0.001]_policy_[0.03 0.03]'
dataset = SeqTeleopDataset(f'../data/lqr_optimal/{setting}', index=[0, 1])
dataloader = DataLoader(dataset, batch_size=batch_size)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training

In [ ]:
model = TransformerHIM(
    d_model=D,
    d_out=O
)

In [ ]:
logger = TensorBoardLogger(
    save_dir=Path('../logs'),
    name=setting,
)

lr_callback = LearningRateMonitor(
    logging_interval="epoch"
)

In [ ]:
trainer = L.Trainer(
    max_epochs=max_epochs,
    devices=[0],
    logger=logger,
    callbacks=[
        lr_callback
    ],
)

In [ ]:
trainer.fit(model, dataloader)

# Inference

In [ ]:
ckpt_path = sorted(Path(f'../logs/{setting}').rglob("*.ckpt"))[-1]
print(f"Loading model from checkpoint: {ckpt_path}")

model = TransformerHIM.load_from_checkpoint(
    ckpt_path,
    d_model=D,
    d_out=O,
    map_location='cpu'
)

In [ ]:
data = next(iter(dataloader))
states, actions, states_next = data
x, x_goal = torch.chunk(states, 2, dim=-1)

In [ ]:
u_H: torch.Tensor = actions.squeeze(0)
u_H_star = model.predict_step(data, 0).squeeze(0)

In [ ]:
# u_H_star = torch.clamp(
#     u_H_star,
#     min=u_H.min(dim=0, keepdim=True).values,
#     max=u_H.max(dim=0, keepdim=True).values
# )

In [ ]:
u_H = u_H.numpy(force=True)
u_H_star = u_H_star.numpy(force=True)

In [ ]:
# u_H_star *= 10

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# Plot first dimension
ax1.plot(u_H[:, 0], label=r'$u_H$', alpha=0.7)
ax1.plot(u_H_star[:, 0], label=r'$u_H^\ast$', alpha=0.7)
ax1.set_ylabel('X')
ax1.grid(True)

# Plot second dimension (stacked on same subplot)
ax2.plot(u_H[:, 1], label=r'$u_H$', alpha=0.7)
ax2.plot(u_H_star[:, 1], label=r'$u_H^\ast$', alpha=0.7)
ax2.set_ylabel('Y')
ax2.set_xlabel('Time Step')
ax2.legend(loc='lower right')
ax2.grid(True)

plt.tight_layout()
plt.show()

# Plot u_H and u_H_star

In [ ]:
distance = []
for idx, d in enumerate(dataloader):
    states, actions, states_next = d
    x, x_goal = torch.chunk(states, 2, dim=-1)
    u_H = actions.squeeze(0)
    u_H_star = model.predict_step(d, idx).squeeze(0)
    
    distance.append(torch.norm(u_H - u_H_star, dim=1).numpy(force=True))

In [ ]:
min_len = min(len(d) for d in distance)
distance = [d[:min_len] for d in distance]
distance = np.stack(distance, axis=0)

In [ ]:
plt.figure(figsize=(10, 6))

# Plot all trajectories as light lines
for i in range(distance.shape[0]):
    plt.plot(distance[i, :], alpha=0.1, color='blue')

# Calculate and plot mean
mean_distance = distance.mean(axis=0)

# Calculate min and max for the range
min_distance = distance.min(axis=0)
max_distance = distance.max(axis=0)

# Fill the range
plt.fill_between(range(len(mean_distance)), min_distance, max_distance, alpha=0.3, color='blue', label='Range')

# Plot mean line
plt.plot(mean_distance, color='red', linewidth=2, label='Mean')

plt.xlabel('Time Step')
plt.ylabel('Distance')
plt.title('Distance between Actual and Optimal Actions')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()